### Imports

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login

### Hugging Face Authentication

In [2]:
# Import token
load_dotenv()

# Login to Hugging Face Hub
login(token=os.getenv('HUGGINGFACE_TOKEN'))

### Functions for Data Exploration

In [3]:
import random as rnd

def random_sample(df, column, label, column_row, range=100):
    """
    Inputs: Dataframe. Column. A specific label to look for in that column. The column you want to return from the result, or you can return all columns/rows. The range for random numbers
    Outputs: A random sample from the dataframe containing the label under a specific column
    """
    if column_row:
        return df.loc[df[str(column)] == str(label)][str(column_row)].iloc[rnd.randrange(0, range)]
    else:
        return df.loc[df[str(column)] == str(label)].iloc[rnd.randrange(0, range)]

### Dataset 1 (Exploration/Pre-processing)

In [4]:
# Read in the dataset
df = pd.read_json("hf://datasets/imnim/multiclass-email-classification/email_dataset.json")

In [ ]:
# Print the first few rows of the dataset
df.head()

,subject,body,labels
0,Meeting Reminder: Quarterly Sales Review Tomorrow,"Dear Team, Just a friendly reminder that our Q...","[Business, Reminders]"
1,Meeting Confirmation for Tomorrow,"Dear Team, This is just a friendly reminder th...","[Business, Events & Invitations]"
2,Important Update: New Company Policies,"Dear Team, I hope this email finds you well. I...",[Business]
3,Important Update: Project Deadline Extension,"Dear Team, I hope this message finds you well....","[Business, Reminders]"
4,Monthly Finance Report,"Dear Team, Please find attached the monthly fi...","[Business, Finance & Bills]"


We seem to have multi-label emails. I will be dropping these and focus on single-label classification due to a lack of data

In [6]:
# Print the distribution of labels in the dataset
df.labels.value_counts().explode().sum

<bound method Series.sum of labels
[Business, Events & Invitations]                      355
[Travel & Bookings]                                   266
[Finance & Bills]                                     222
[Business, Reminders]                                 174
[Personal]                                            113
[Job Application]                                     105
[Business, Customer Support]                          103
[Newsletters]                                         103
[Promotions]                                          102
[Business, Events & Invitations, Reminders]            82
[Events & Invitations]                                 65
[Personal, Newsletters]                                46
[Events & Invitations, Business, Reminders]            44
[Events & Invitations, Business]                       39
[Business, Customer Support, Finance & Bills]          37
[Business, Events & Invitations, Newsletters]          32
[Customer Support, Finance & Bills]  

In [7]:
# Print all unique labels in the dataset
unique_elements = df['labels'].explode().unique()
print(unique_elements)

<ArrowStringArray>
[            'Business',            'Reminders', 'Events & Invitations',
      'Finance & Bills',    'Travel & Bookings',     'Customer Support',
          'Newsletters',             'Personal',      'Job Application',
           'Promotions']
Length: 10, dtype: str


#### Features that will be used for the model:
    - Business (Rename to 'Work')
    - Job Application (Rename to 'Work')
    - Events & Invitations
    - Finance & Bills
    - Travel & Bookings
    - Newsletters
    - Personal
    - Promotions
    - Customer Support (Rename to 'Updates')

In [8]:
# Function that converts multi-label to single-label based on label priority
def clean_labels(category_list):
    # Priority for converting multi-label to single-label
    label_priority = {
        'Business' : 0, 
        'Job Application' : 1,
        'Personal' : 2,
        'Finance & Bills' : 3,
        'Events & Invitations' : 4, 
        'Travel & Bookings' : 5,
        'Customer Support' : 6,
        'Newsletters': 7,
        'Promotions' : 7
    }

    # Only keep categories we want, drop any others 
    category_list = [i for i in category_list if i in label_priority]
    
    # If we have more than one valid category, sort them according to our priority
    if len(category_list) > 0:
        category_list = sorted(category_list, key=lambda x: label_priority[x])
    else:
        return None
    
    return category_list[0]

In [9]:
# Clean the labels
df['labels'] = df['labels'].apply(clean_labels)

# Drop any rows that don't have category labels
df.dropna(inplace=True)

# Conversion for 'Work'
work_dict = {
    'Business' : 'Work',
    'Job Application': 'Work'
}

# Conversion for 'Updates'
updates_dict = {
    'Customer Support' : 'Updates'
}

# Rename certain labels to 'Work' and 'Updates' respecetively
df['labels'] = df['labels'].replace(work_dict)
df['labels'] = df['labels'].replace(updates_dict)

In [10]:
df.labels.value_counts()

labels
Work                    1063
Travel & Bookings        282
Finance & Bills          261
Personal                 204
Promotions               109
Newsletters              105
Events & Invitations      75
Updates                    6
Name: count, dtype: int64

## Dataset 2

In [11]:
splits = {'train': 'train.json', 'test': 'test.json'}
df2_train = pd.read_json("hf://datasets/jason23322/high-accuracy-email-classifier/" + splits["train"])
df2_test = pd.read_json("hf://datasets/jason23322/high-accuracy-email-classifier/" + splits["test"])
df2 = pd.merge(df2_train, df2_test, how='outer')

In [12]:
df2.head()

,id,subject,body,text,category,category_id
0,forum_1,[r/techsupport] Reply: Slow boot issue,User: PC_Enthusiast23 suggested BIOS update mi...,[r/techsupport] Reply: Slow boot issue User: P...,forum,0
1,forum_10,"Your post was moved to ""Technical Support""","Thread ""Network issues"" moved from General to ...","Your post was moved to ""Technical Support"" Thr...",forum,0
2,forum_100,Warning: Unofficial Discord Server,"Beware of phishing links in ""Forum Chat"" posts...",Warning: Unofficial Discord Server Beware of p...,forum,0
3,forum_1000,Mod application open for Technical Support,"Thread: ""Discussion 870"" started by DataWiz. S...",Mod application open for Technical Support Thr...,forum,0
4,forum_1001,AMA session: Tech CEO,"Thread ""Discussion 610"" moved from General to ...","AMA session: Tech CEO Thread ""Discussion 610"" ...",forum,0


##### Notes About the Data
- **verify_code**: Labels seem to be about authentication such as temporary pins provided for signing in or reseting password
    - This could be used for 'Updates' column, but I would have to sample less due to 1:375 ratio I have with 'Updates' data
- **forum**: Labels don't seem useful for this task, will be dropped
- **promotions**: Seem on par with my first data frame's 'Promotions' label, however, the first dataframe seem to be more rich in text
- **social_media**: Will be used
- **updates**: Will be used

In [13]:
# Only keep 'subject', 'body', and 'category'
df2 = df2.drop(['id', 'text', 'category_id'], axis=1)

# Rename 'category' to 'labels'
df2.rename(columns={'category':'labels'}, inplace=True)

# Rename 'verify_code' to 'Updates'
df2['labels'] = df2['labels'].replace({'verify_code':'Updates'})

# Drop any labels that have 'forum' entries
df2['labels'] = df2['labels'].apply(lambda x: None if x == 'forum' else x)
df2.dropna(inplace=True)

In [14]:
df2.labels.value_counts()

labels
Updates         2251
promotions      2245
social_media    2245
spam            2243
updates         2243
Name: count, dtype: int64

In [15]:
df.labels.value_counts()

labels
Work                    1063
Travel & Bookings        282
Finance & Bills          261
Personal                 204
Promotions               109
Newsletters              105
Events & Invitations      75
Updates                    6
Name: count, dtype: int64

In [318]:
# Print random samples from specific labels
random_sample(df2, 'labels', 'spam', 'body', 1000)

'Your $5000 refund is processed. Claim: bit.ly/fakeprize Complete within 2hrshrs.'